# Building an Interactive Customer Segmentation Dashboard with Plotly Dash

**SIADS 521 - Assignment 3**  
**Author:** Brett Wolff

---

In this tutorial, we build an interactive, multi-panel dashboard that explores the *Mall Customers* dataset. The dashboard combines **scatter plots, bar charts, histograms, box plots, and a heatmap** into a single screen where every filter simultaneously updates every visualization. By the end of this notebook you will be able to:

1. Describe when each visualization type is most useful.
2. Set up Plotly and Dash from scratch.
3. Clean and prepare a dataset for dashboard consumption.
4. Build individual Plotly figures and wire them together inside a Dash application with cross-filtering callbacks.
5. Deploy the finished dashboard 

---

# 1. Visualization Technique

This section describes **each visualization type** used in the dashboard, explains how they complement one another, and discusses the interactivity features that tie them together.

## 1.1 Scatter Plot — *Income vs. Spending Score*

A scatter plot encodes two continuous variables as x/y positions and optionally maps a categorical variable to color. In our dashboard, **Annual Income** sits on the x-axis and **Spending Score** on the y-axis, with points colored by **Gender**.

**When to use it:** Whenever you need to reveal the *relationship* between two numeric variables, spot clusters, or detect outliers. Our scatter plot is the centerpiece of the dashboard because it immediately surfaces the five natural customer segments in the Mall Customers data (e.g., high-income/high-spending "VIPs" versus low-income/high-spending "careful spenders").

## 1.2 Bar Chart — *Average Spending & Income by Age Group*

A grouped bar chart compares aggregated measures across categories. We bin customers into age groups (18–25, 26–35, 36–45, 46–55, 56–70) and display the **mean Annual Income** and **mean Spending Score** side by side for each bin.

**When to use it:** For comparing a summary statistic across discrete categories. The grouped layout makes it easy to spot that younger customers (18–25) tend to have lower incomes but spend at above-average rates, a pattern that is harder to notice in a scatter plot alone.

## 1.3 Histogram — *Age Distribution*

A histogram visualizes the *frequency distribution* of a single continuous variable by partitioning values into equal-width bins. We stack the histogram by **Gender** so the audience can simultaneously read the overall age distribution and the male/female breakdown within each bin.

**When to use it:** To understand the shape, center, and spread of a single variable. Our histogram reveals that the mall's customer base is concentrated between ages 25–40, with a roughly symmetric distribution.

## 1.4 Box Plot — *Income & Spending Distributions by Gender*

Box plots display the five-number summary (min, Q1, median, Q3, max) plus outliers. We render two box plots side by side: one for **Annual Income** and one for **Spending Score**, both split by **Gender**.

**When to use it:** To compare distributions across groups and to flag outliers. In our data the box plots show that male and female customers have similar income distributions, but female customers exhibit a slightly wider spread in spending scores.

## 1.5 Heatmap — *Correlation Matrix*

A heatmap renders values in a matrix as colored cells. We compute the Pearson correlation between Age, Annual Income, and Spending Score and display it as an annotated color grid.

**When to use it:** To quickly scan for linear relationships among many variables at once. The heatmap confirms that income and spending score are only weakly correlated overall (a relationship that becomes clearer when viewed through the scatter plot's clusters).

## 1.6 How These Visualizations Complement Each Other

| Visualization | Strength | What It Adds to the Story |
|---|---|---|
| Scatter plot | Reveals clusters and relationships | Identifies distinct customer segments |
| Bar chart | Aggregated comparison across categories | Connects demographics (age groups) to financial behavior |
| Histogram | Distributional shape | Shows the customer base composition |
| Box plot | Distribution comparison and outlier detection | Highlights gender-based differences |
| Heatmap | Multi-variable correlation at a glance | Validates (or challenges) assumed relationships |

Together these five chart types move the analyst from *"who are our customers?"* (histogram, box plots) through *"how do demographics relate to spending?"* (bar chart, heatmap) to *"what distinct segments exist?"* (scatter plot). No single chart type can tell this full story alone.

## 1.7 Dashboard Interactivity Considerations

Our dashboard implements three interactive controls that **affect all five visualizations simultaneously**:

- **Gender Dropdown** — Filter to Male, Female, or All. Every chart re-renders to show only the selected subset.
- **Age Range Slider** — Restrict the analysis to a minimum-maximum age window. The histogram, bar chart, and scatter plot all update in real time.
- **Hover Tooltips** — Every data point shows contextual detail (e.g., exact income, spending score, and age) on hover without cluttering the static view.

Cross-filtering is the key design pattern: a single callback function in Dash receives all filter values, applies them to the dataframe, and returns updated figures. This ensures that every panel always reflects the same subset of data, preventing the viewer from drawing contradictory conclusions across charts.

---

# 2. Visualization Library

## 2.1 Why Plotly + Dash?

For this assignment we use **Plotly** (for individual chart objects) and **Dash** (for the dashboard framework that hosts them). Here is a quick orientation:

| Aspect | Details |
|---|---|
| **Creator** | Plotly, Inc. (Montreal, Canada), founded 2012 |
| **License** | MIT — fully open source |
| **Language** | Python, R, Julia, and JavaScript bindings |
| **Install** | `pip install dash plotly pandas` (Dash bundles Plotly) |
| **Current versions** | Dash 4.0.0, Plotly 6.6.0 (as used in this project) |

### Why not Altair or Matplotlib?

- **Altair** is declarative and elegant for static exploratory plots, but its interactivity is limited to selections within a single Vega-Lite specification. It cannot drive server-side callbacks that re-query a dataframe.
- **Matplotlib** is procedural and pixel-based; creating interactive web dashboards requires bolting on extra frameworks (e.g., Panel, mpld3) and the resulting experience is not as smooth.
- **Plotly + Dash** is purpose-built for interactive web dashboards: Plotly renders GPU-accelerated charts in the browser via WebGL, while Dash provides a pure-Python callback system that keeps all logic in one place with no JavaScript required.

## 2.2 Declarative vs. Procedural

Plotly occupies a middle ground:

- **Plotly Express** (`plotly.express`) is *declarative*: you describe *what* you want (data, axes, color mapping) and Plotly chooses the rendering details. This is what we use for rapid prototyping of each chart.
- **Plotly Graph Objects** (`plotly.graph_objects`) is *procedural*: you build a `Figure` from explicit `Trace` and `Layout` objects. We drop down to this level when we need fine-grained control (e.g., the correlation heatmap with custom annotations).

Dash itself is **reactive-declarative**: you declare the layout as a tree of HTML/Dash components and then declare callback functions that map input property changes to output property updates. Dash handles all the HTTP plumbing and React rendering under the hood.

## 2.3 Jupyter Integration

Plotly figures render inline in Jupyter notebooks via the `plotly` renderer, so we can prototype every chart right here before porting it to the standalone Dash app. Dash apps themselves run as a local web server (not inside a notebook cell), so the final dashboard is launched with `python app.py` and viewed at `http://127.0.0.1:8050`.

## 2.4 Limitations

- **Server requirement:** Dash requires a running Python process to handle callbacks. Static hosting (GitHub Pages) won't work — you need a server (Render, Heroku, or a local machine).
- **Scalability:** Very large datasets (>100k rows) can slow down browser rendering. Techniques like data aggregation or WebGL scatter (`Scattergl`) mitigate this.
- **Styling:** Dash's default styling is functional but plain; polished UIs may require custom CSS or Dash Bootstrap Components.

---

# 3. Demonstration

## 3.1 Dataset Overview and Cleaning

The **Mall Customers** dataset contains 200 records of shopping-mall visitors. Each row has five columns:

| Column | Type | Description |
|---|---|---|
| CustomerID | int | Unique identifier (1-200) |
| Gender | str | Male or Female |
| Age | int | Customer age (18-70) |
| Annual Income (k$) | int | Yearly income in thousands of dollars (15-137) |
| Spending Score (1-100) | int | Mall-assigned score based on spending behavior (1-99) |

The dataset is clean out of the box — no missing values, no duplicates. We will rename columns for convenience and add an **Age Group** bin that makes bar-chart aggregation cleaner.

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Load the dataset
df = pd.read_csv("Mall_Customers.csv")

# Rename columns for easier access
df.columns = ["CustomerID", "Gender", "Age", "Income", "SpendingScore"]

# Confirm no missing values
print("Missing values:\n", df.isnull().sum())
print("\nShape:", df.shape)
print("\nFirst 5 rows:")
df.head()

Missing values:
 CustomerID       0
Gender           0
Age              0
Income           0
SpendingScore    0
dtype: int64

Shape: (200, 5)

First 5 rows:


,CustomerID,Gender,Age,Income,SpendingScore
0,1,Male,19,15,39
1,2,Male,21,15,81
2,3,Female,20,16,6
3,4,Female,23,16,77
4,5,Female,31,17,40


In [2]:
# Create Age Group bins for the bar chart
bins = [17, 25, 35, 45, 55, 70]
labels = ["18-25", "26-35", "36-45", "46-55", "56-70"]
df["AgeGroup"] = pd.cut(df["Age"], bins=bins, labels=labels)

print("Age Group distribution:")
print(df["AgeGroup"].value_counts().sort_index())
print("\nSummary statistics:")
df.describe()

Age Group distribution:
AgeGroup
18-25    38
26-35    60
36-45    36
46-55    37
56-70    29
Name: count, dtype: int64

Summary statistics:


,CustomerID,Age,Income,SpendingScore
count,200.000000,200.000000,200.000000,200.000000
mean,100.500000,38.850000,60.560000,50.200000
std,57.879185,13.969007,26.264721,25.823522
min,1.000000,18.000000,15.000000,1.000000
25%,50.750000,28.750000,41.500000,34.750000
50%,100.500000,36.000000,61.500000,50.000000
75%,150.250000,49.000000,78.000000,73.000000
max,200.000000,70.000000,137.000000,99.000000


## 3.2 Building Individual Visualizations

We will build each chart one at a time using **Plotly Express** (and Plotly Graph Objects for the heatmap). This lets us verify every visualization before combining them into the final Dash dashboard.

### 3.2.1 Scatter Plot: Income vs. Spending Score

The scatter plot places Annual Income on the x-axis and Spending Score on the y-axis, with color encoding Gender. 

1. **Low income, low spending** (bottom-left)
2. **Low income, high spending** (top-left)
3. **Medium income, medium spending** (center)
4. **High income, low spending** (bottom-right)
5. **High income, high spending** (top-right)


In [3]:
# Scatter Plot: Income vs. Spending Score
fig_scatter = px.scatter(
    df,
    x="Income",
    y="SpendingScore",
    color="Gender",
    hover_data=["Age", "CustomerID"],
    title="Annual Income vs. Spending Score",
    labels={"Income": "Annual Income (k$)", "SpendingScore": "Spending Score (1-100)"},
    color_discrete_map={"Male": "#636EFA", "Female": "#EF553B"},
    opacity=0.7,
)
fig_scatter.update_layout(template="plotly_white", height=450)
fig_scatter.show()

The five clusters are clearly visible. Notice that gender is fairly evenly distributed across all clusters. The segmentation is driven by income and spending behavior, not gender.

### 3.2.2 Grouped Bar Chart: Average Income & Spending by Age Group

Next we aggregate by age group to see how financial behavior varies across demographics. We compute the mean Income and mean Spending Score per age bin, then use a grouped bar chart to compare them side by side.

In [4]:
# Grouped Bar Chart: Average Income & Spending Score by Age Group
agg = df.groupby("AgeGroup", observed=True)[["Income", "SpendingScore"]].mean().reset_index()

fig_bar = go.Figure()
fig_bar.add_trace(go.Bar(
    x=agg["AgeGroup"], y=agg["Income"],
    name="Avg Income (k$)", marker_color="#636EFA"
))
fig_bar.add_trace(go.Bar(
    x=agg["AgeGroup"], y=agg["SpendingScore"],
    name="Avg Spending Score", marker_color="#EF553B"
))
fig_bar.update_layout(
    barmode="group",
    title="Average Income & Spending Score by Age Group",
    xaxis_title="Age Group",
    yaxis_title="Value",
    template="plotly_white",
    height=400,
)
fig_bar.show()

The bar chart reveals that **income peaks in the 26-45 age range** while **spending scores are relatively stable across age groups**, with a slight decline for the oldest cohort (56-70). This suggests that older, higher-income customers are not necessarily the biggest spenders. This is an insight a marketing team can act on.

### 3.2.3 Histogram: Age Distribution by Gender

A stacked histogram shows the overall shape of the age distribution and the gender breakdown within each bin.

In [5]:
# Histogram: Age Distribution by Gender
fig_hist = px.histogram(
    df,
    x="Age",
    color="Gender",
    nbins=15,
    barmode="stack",
    title="Age Distribution by Gender",
    labels={"Age": "Age (years)", "count": "Number of Customers"},
    color_discrete_map={"Male": "#636EFA", "Female": "#EF553B"},
)
fig_hist.update_layout(template="plotly_white", height=400)
fig_hist.show()

The histogram shows a **roughly symmetric distribution centered around age 30-35**, with female customers outnumbering male customers in most bins. The mall's customer base skews young-to-middle-aged, with relatively few visitors over 55.

### 3.2.4 Box Plot — Income and Spending Score by Gender

Box plots let us compare distributional characteristics (median, spread, outliers) between male and female customers for both financial variables.

In [6]:
# Box Plots: Income and Spending Score by Gender (side by side)
fig_box = make_subplots(rows=1, cols=2, subplot_titles=("Annual Income by Gender", "Spending Score by Gender"))

for i, col in enumerate(["Income", "SpendingScore"], 1):
    for gender, color in [("Male", "#636EFA"), ("Female", "#EF553B")]:
        subset = df[df["Gender"] == gender]
        fig_box.add_trace(
            go.Box(y=subset[col], name=gender, marker_color=color,
                   showlegend=(i == 1)),
            row=1, col=i,
        )

fig_box.update_layout(
    title="Income & Spending Score Distributions by Gender",
    template="plotly_white",
    height=400,
)
fig_box.show()

The box plots confirm that **income distributions are nearly identical** between genders (similar medians around $60k, similar IQRs). 

### 3.2.5 Heatmap: Correlation Matrix

Finally, we compute the Pearson correlation matrix for the three numeric variables and visualize it as an annotated heatmap. This confirms relationships we have already glimpsed in the other charts.

In [7]:
# Heatmap: Correlation Matrix
corr_cols = ["Age", "Income", "SpendingScore"]
corr_matrix = df[corr_cols].corr()

fig_heat = go.Figure(data=go.Heatmap(
    z=corr_matrix.values,
    x=corr_cols,
    y=corr_cols,
    text=np.round(corr_matrix.values, 2),
    texttemplate="%{text}",
    colorscale="RdBu_r",
    zmin=-1, zmax=1,
))
fig_heat.update_layout(
    title="Correlation Matrix: Age, Income, Spending Score",
    template="plotly_white",
    height=400,
    width=500,
)
fig_heat.show()

The heatmap confirms **weak linear correlations** between all three variables. The near-zero correlation between Income and Spending Score is particularly interesting. It tells us that high earners are not automatically high spenders (and vice versa), which is exactly what the scatter plot's five-cluster structure shows us visually.

---

## 3.3 Building the Interactive Dashboard with Dash

Now that we have verified each visualization individually, we combine them into a single interactive Dash application. The complete source code lives in `app.py` in this directory. Below is a walkthrough of the key components.

### 3.3.1 App Layout

The Dash layout is a tree of HTML and Dash Core Components. We use a two-column grid: **filters on the left**, **charts on the right** arranged in a 2x2 + 1 grid (scatter + bar on row 1, histogram + box on row 2, heatmap centered on row 3).

```python
app.layout = html.Div([
    # Header
    html.H1("Mall Customer Segmentation Dashboard"),

    # Filters row
    html.Div([
        # Gender dropdown
        dcc.Dropdown(id="gender-filter", options=[...], value="All"),
        # Age range slider
        dcc.RangeSlider(id="age-slider", min=18, max=70, value=[18, 70]),
    ]),

    # Charts grid
    html.Div([
        dcc.Graph(id="scatter-plot"),
        dcc.Graph(id="bar-chart"),
        dcc.Graph(id="histogram"),
        dcc.Graph(id="box-plot"),
        dcc.Graph(id="heatmap"),
    ]),
])
```

### 3.3.2 Callbacks — Cross-Filtering

The magic of Dash is the **callback decorator**. A single function receives the current filter values and returns all five updated figures:

```python
@app.callback(
    [Output("scatter-plot", "figure"),
     Output("bar-chart", "figure"),
     Output("histogram", "figure"),
     Output("box-plot", "figure"),
     Output("heatmap", "figure")],
    [Input("gender-filter", "value"),
     Input("age-slider", "value")]
)
def update_dashboard(gender, age_range):
    # Filter the dataframe
    filtered = df[(df["Age"] >= age_range[0]) & (df["Age"] <= age_range[1])]
    if gender != "All":
        filtered = filtered[filtered["Gender"] == gender]

    # Build all five figures from the filtered data
    # ... (same Plotly code from above, but using `filtered` instead of `df`)
    return fig_scatter, fig_bar, fig_hist, fig_box, fig_heat
```

This pattern ensures that **every chart always reflects the same filtered subset**. When the user moves the age slider from 18-70 to 25-40, all five charts instantly update to show only that age window.

### 3.3.3 Running the Dashboard Locally

To launch the dashboard:

```bash
conda activate siads521-dashboard
python app.py
```

Then open **http://127.0.0.1:8050** in your browser. The full `app.py` source code is included in this submission.

## 3.4 Full Interactive Dashboard (In-Notebook)

The cell below launches the **complete interactive dashboard directly inside this notebook**. It uses Dash's built-in `jupyter_mode="inline"` to embed the app as an iframe in the cell output. All five visualizations appear on a single screen, and the **Gender dropdown** and **Age Range slider** cross-filter every chart simultaneously.

> **Note:** If the dashboard does not render, try restarting the kernel and running **all cells from the top**. If it still shows blank, run `!pip install dash==3.2.0 plotly comm` and restart the kernel — Dash 3.3+ has a known inline-mode bug in JupyterHub environments.

In [ ]:
# ---------------------------------------------------------------------------
# Full Interactive Dashboard — rendered inline in this notebook
# ---------------------------------------------------------------------------
# Install comm package if missing (required for Dash inline mode)
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "comm"])

from dash import Dash, dcc, html, Input, Output
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd
import numpy as np

# Self-contained: load and prepare data so this cell works after kernel restart
df = pd.read_csv("Mall_Customers.csv")
df.columns = ["CustomerID", "Gender", "Age", "Income", "SpendingScore"]
bins = [17, 25, 35, 45, 55, 70]
labels = ["18-25", "26-35", "36-45", "46-55", "56-70"]
df["AgeGroup"] = pd.cut(df["Age"], bins=bins, labels=labels)

COLOR_MAP = {"Male": "#636EFA", "Female": "#EF553B"}

# Create the Dash app with Vocareum/JupyterHub proxy prefix
app = Dash(__name__, requests_pathname_prefix="/proxy/8050/")

app.layout = html.Div(
    style={"fontFamily": "Segoe UI, Arial, sans-serif", "padding": "10px"},
    children=[
        html.H2("Mall Customer Segmentation Dashboard",
                 style={"textAlign": "center", "marginBottom": "5px"}),
        html.P("Use the filters below to cross-filter all five visualizations simultaneously.",
               style={"textAlign": "center", "color": "#666", "marginBottom": "15px"}),

        # --- Filters ---
        html.Div(
            style={"display": "flex", "gap": "30px", "alignItems": "center",
                   "marginBottom": "20px", "flexWrap": "wrap",
                   "background": "#f8f9fa", "padding": "12px 16px",
                   "borderRadius": "8px"},
            children=[
                html.Div([
                    html.Label("Gender:", style={"fontWeight": "bold", "marginRight": "8px"}),
                    dcc.Dropdown(
                        id="gender-filter",
                        options=[{"label": g, "value": g} for g in ["All", "Male", "Female"]],
                        value="All", clearable=False,
                        style={"width": "140px"},
                    ),
                ], style={"display": "flex", "alignItems": "center"}),
                html.Div([
                    html.Label("Age Range:", style={"fontWeight": "bold", "marginRight": "10px"}),
                    dcc.RangeSlider(
                        id="age-slider",
                        min=18, max=70, step=1, value=[18, 70],
                        marks={i: str(i) for i in range(18, 71, 10)},
                        tooltip={"placement": "bottom", "always_visible": False},
                    ),
                ], style={"flex": "1", "minWidth": "250px"}),
            ],
        ),

        # --- Row 1: Scatter + Bar ---
        html.Div(style={"display": "flex", "gap": "10px", "flexWrap": "wrap"}, children=[
            html.Div(dcc.Graph(id="nb-scatter"), style={"flex": "1", "minWidth": "380px"}),
            html.Div(dcc.Graph(id="nb-bar"),     style={"flex": "1", "minWidth": "380px"}),
        ]),
        # --- Row 2: Histogram + Box ---
        html.Div(style={"display": "flex", "gap": "10px", "flexWrap": "wrap"}, children=[
            html.Div(dcc.Graph(id="nb-hist"), style={"flex": "1", "minWidth": "380px"}),
            html.Div(dcc.Graph(id="nb-box"),  style={"flex": "1", "minWidth": "380px"}),
        ]),
        # --- Row 3: Heatmap (centered) ---
        html.Div(style={"display": "flex", "justifyContent": "center"}, children=[
            html.Div(dcc.Graph(id="nb-heat"), style={"width": "480px"}),
        ]),
    ],
)


@app.callback(
    [Output("nb-scatter", "figure"),
     Output("nb-bar",     "figure"),
     Output("nb-hist",    "figure"),
     Output("nb-box",     "figure"),
     Output("nb-heat",    "figure")],
    [Input("gender-filter", "value"),
     Input("age-slider",    "value")],
)
def update_all(gender, age_range):
    filt = df[(df["Age"] >= age_range[0]) & (df["Age"] <= age_range[1])]
    if gender != "All":
        filt = filt[filt["Gender"] == gender]

    # 1 — Scatter
    fig1 = px.scatter(filt, x="Income", y="SpendingScore", color="Gender",
                      hover_data=["Age", "CustomerID"],
                      title="Annual Income vs. Spending Score",
                      labels={"Income": "Annual Income (k$)",
                              "SpendingScore": "Spending Score (1-100)"},
                      color_discrete_map=COLOR_MAP, opacity=0.7)
    fig1.update_layout(template="plotly_white", height=370, margin=dict(t=40, b=30))

    # 2 — Bar
    agg = filt.groupby("AgeGroup", observed=True)[["Income", "SpendingScore"]].mean().reset_index()
    fig2 = go.Figure()
    fig2.add_trace(go.Bar(x=agg["AgeGroup"], y=agg["Income"],
                          name="Avg Income (k$)", marker_color="#636EFA"))
    fig2.add_trace(go.Bar(x=agg["AgeGroup"], y=agg["SpendingScore"],
                          name="Avg Spending Score", marker_color="#EF553B"))
    fig2.update_layout(barmode="group",
                       title="Avg Income & Spending by Age Group",
                       xaxis_title="Age Group", yaxis_title="Value",
                       template="plotly_white", height=370, margin=dict(t=40, b=30))

    # 3 — Histogram
    fig3 = px.histogram(filt, x="Age", color="Gender", nbins=15, barmode="stack",
                        title="Age Distribution by Gender",
                        labels={"Age": "Age (years)"},
                        color_discrete_map=COLOR_MAP)
    fig3.update_layout(template="plotly_white", height=370, margin=dict(t=40, b=30))

    # 4 — Box plots
    fig4 = make_subplots(rows=1, cols=2,
                         subplot_titles=("Income by Gender", "Spending Score by Gender"))
    for i, col in enumerate(["Income", "SpendingScore"], 1):
        for g, c in [("Male", "#636EFA"), ("Female", "#EF553B")]:
            sub = filt[filt["Gender"] == g]
            fig4.add_trace(go.Box(y=sub[col], name=g, marker_color=c,
                                  showlegend=(i == 1)), row=1, col=i)
    fig4.update_layout(title="Distributions by Gender",
                       template="plotly_white", height=370, margin=dict(t=50, b=30))

    # 5 — Heatmap
    cols = ["Age", "Income", "SpendingScore"]
    corr = filt[cols].corr() if len(filt) >= 2 else pd.DataFrame(
        np.zeros((3, 3)), index=cols, columns=cols)
    fig5 = go.Figure(data=go.Heatmap(
        z=corr.values, x=cols, y=cols,
        text=np.round(corr.values, 2), texttemplate="%{text}",
        colorscale="RdBu_r", zmin=-1, zmax=1))
    fig5.update_layout(title="Correlation Matrix",
                       template="plotly_white", height=370, width=460,
                       margin=dict(t=40, b=30))

    return fig1, fig2, fig3, fig4, fig5


# Launch inside the notebook (works on Vocareum / JupyterHub)
app.run(jupyter_mode="inline", port=8050, jupyter_height=900)

---

# 4. Deployment Instructions

## 4.1 Running Locally

**Prerequisites:** [Anaconda](https://www.anaconda.com/download) or [Miniconda](https://docs.conda.io/en/latest/miniconda.html) installed.

**Step 1 Create and activate the conda environment:**

```bash
conda create -n siads521-dashboard python=3.11 -y
conda activate siads521-dashboard
```

**Step 2 Install dependencies:**

```bash
pip install -r assignment_3_requirements.txt
```

Or install manually:

```bash
pip install dash==4.0.0 plotly==6.6.0 pandas==3.0.1 numpy==2.4.3
```

**Step 3 Launch the dashboard:**

```bash
python app.py
```

**Step 4 Open in browser:**

Navigate to **http://127.0.0.1:8050**. The dashboard will display all five visualizations with interactive filters.

```

---

# 5. Troubleshooting Common Errors

## 5.1 `Address already in use` when running `app.py`

**Cause:** Another process (or a previous Dash instance) is already bound to port 8050.

**Fix:** Kill the existing process or use a different port:
```bash
# Find and kill the process on port 8050
lsof -ti:8050 | xargs kill -9

# Or run on a different port
python app.py  # then change the port in app.py's app.run() call
```

## 5.2 Plotly figures appear blank in the notebook

**Cause:** The notebook renderer is not configured. In some environments Plotly defaults to a renderer that does not work in Jupyter.

**Fix:** Add this at the top of your notebook:
```python
import plotly.io as pio
pio.renderers.default = "notebook"
```

---

# 6. Conclusions

Our interactive dashboard tells a cohesive story about mall customer segmentation:

1. **The customer base is young and predominantly female**  the histogram shows most visitors are aged 25-40, with women outnumbering men in nearly every age bin.
2. **Income and spending behavior are surprisingly uncorrelated** — the heatmap shows a near-zero Pearson r, and the scatter plot reveals why: customers cluster into five distinct segments rather than following a simple linear trend.
3. **Age affects income more than spending** — the bar chart shows income peaking in middle age groups while spending scores remain relatively flat, suggesting that younger customers punch above their income weight in spending.
4. **Gender differences are minimal** — the box plots confirm that male and female customers have nearly identical income and spending distributions.
5. **Cross-filtering enriches the analysis** — by slicing the data with the gender dropdown and age slider, we can drill into specific subpopulations (e.g., "young females with high spending scores") and verify that these patterns hold across segments.

These insights demonstrate the power of combining multiple visualization types in an interactive dashboard: no single chart could reveal all five conclusions, but together they provide a complete, actionable picture of the customer base.